# Logistic birth rate - Inference Quality Analyses

In this notebook we use synthetic viral read count data from a fully-parameterised toy population to theoretically assess
 - (1) the practical identifiability and 
 - (2) the quality of parameter inference

when the rodent population are assumed to follow the dynamics of the SIR algorithm with a logistic birth term rate. If the estimates of the population model parameters are close to the true model parameter values that characterise the toy population in the first place, this implies the validity of inferential approach, and therefore lend credibility to the results produced when the same pipeline is applied to metaviromic datasets collected through wildlife studies.

Similar to field studies, random samples of rodents are drawn from the simulated toy population at predifined sampling times, which satisfy the following:
 - same total number of rodents sampled at each time point;
 - the sampled individuals can be either susceptible (S), infected (I) or recovered (R), with no predefined quantities of each;
 - all individuals sampled are born and alive at the time of sampling.

For each of the sampled individuals, we use the SIR model's embedded `ct_model` to produce Ct value data, similar to what data is produced from the field studies (byproduct in our analyses, ground truth in real studies).

For the parameter inference we follow an optimisation approach, using the Bare-bones CMA-ES method from *Pints [2]* for single parameter inference.

We replicate these analyses for a range of sample sizes and frequencies of sampling values, to compare the quality of parameter estimation and proportion of infected population across different sampling protocols.

**************
### References
[1] James A. Hay et al., _Estimating epidemiologic dynamics from cross-sectional viral load distributions_. Science373,**eabh0635(2021)**. DOI:10.1126/science.abh0635

[2] Clerx, M., Robinson, M., Lambert, B., Lei, C. L., Ghosh, S., Mirams, G. R., & Gavaghan, D. J.,
_Probabilistic Inference on Noisy Time Series (PINTS)_.
Journal of Open Research Software (2019), 7(1), 23. DOI:10.5334/jors.252

In [1]:
# Load necessary libraries
import numpy as np
import pandas as pd
from scipy.stats import multinomial, skew, gumbel_r
import math
import metavirommodel as mm
import metavirommodel.inference as mmi
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pints
from matplotlib import pyplot as plt
import pints.plot

# Choose array of colours for graphs and compartments names
colours = ['blue', 'red', 'green', 'purple', 'orange', 'black', 'gray', 'pink']
compartments = ['S', 'I', 'R']

# Set random seed
np.random.seed(270)

## Gillespie stochastic SIR algorithm with logistic birth term rate

#### Define rodent population

In [2]:
# Set initial reproduction number
R_0 = 3

# Set initial population state S - I - R
N_init = 200
# S_init = int(N_init / R_0)
S_init = 180
I_init = N_init - S_init
R_init = 0
initial_population = [S_init, I_init, R_init]

# Set birth rate
theta = 0.05

# Set death rates
mu = 0.002
nu = 0

# Set transition rates
infect_period = 30
beta =  R_0 / infect_period
gamma = 1 / infect_period

# Coalesce into paramater vector
parameters = initial_population
parameters.extend([theta, mu, nu, beta, gamma])

# Instantiate algorithm
algorithm = mm.LogisticGrowthMetaviromodel(carrying_capacity=400)

# Select start and end times
start_time = 1
end_time = 360

times = list(range(start_time, end_time+1))

# Select number of experiments
num_experiments = 1

output_algorithm = []

S_history_algorithm = []
I_history_algorithm = []
R_history_algorithm = []

I_times_history_algorithm = []
R_times_history_algorithm = []

for _ in range(num_experiments):
    output, S_history, I_history, R_history, I_times_history, R_times_history = algorithm.simulate_fixed_times(parameters, start_time, end_time)
    output_algorithm.append(output)

    S_history_algorithm.append(S_history)
    I_history_algorithm.append(I_history)
    R_history_algorithm.append(R_history)

    I_times_history_algorithm.append(I_times_history)
    R_times_history_algorithm.append(R_times_history)

output_algorithm = np.asarray(output_algorithm)

### Plot output of Gillespie for the different compartments

In [3]:
# Trace names - represent the type of individuals for the simulation
trace_name = ['{}'.format(s) for s in compartments]

# Names of panels
panels = ['{} only'.format(s) for s in compartments] + ['Total Population']

fig = go.Figure()
fig = make_subplots(rows=int(np.ceil(len(panels)/2)), cols=2, subplot_titles=tuple('{}'.format(p) for p in panels))

# Add traces to the separate counts panels
for s, spec in enumerate(compartments):
    fig.add_trace(
        go.Scatter(
            y=np.mean(output_algorithm[:, :, s], axis=0).tolist(),
            x=times,
            mode='lines',
            name=trace_name[s],
            line_color=colours[s]
        ),
        row= int(np.floor(s / 2)) + 1,
        col= s % 2 + 1
    )

fig.add_trace(
    go.Scatter(
        y=np.mean(np.sum(output_algorithm, axis=2), axis=0).tolist(),
        x=times,
        mode='lines',
        name='Total Population',
        line_color='black'
    ),
    row= 2,
    col= 2
)

# Add axis labels
fig.update_layout(
    title='Counts of compartments over time:<br>IC = {}, θ = {}, μ = {}, v = {}, β = {:.2f}, γ = {:.2f}'.format(parameters[0:3], parameters[3], parameters[4], parameters[5], parameters[6], parameters[7]),
    width=1100, 
    height=600,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis2=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis2=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis3=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis3=dict(
        linecolor='black',
        title = 'Individuals'),
    xaxis4=dict(
        linecolor='black',
        title = 'Time (days)'
        ),
    yaxis4=dict(
        linecolor='black',
        title = 'Individuals')
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Logistic_SIR-gillespie.pdf')
fig.show()

## Produce Ct values

In [4]:
# Set parameter for the viral read counts model
t_eclipse = 3  # (0 days) Time from infection to initial viral growth
t_peak = 7  # (5 days ) Time from initial viral growth to peak viral load
t_switch = 5  # (9.38 days) Time from peak viral load to secondary waning phase
t_mod = 15  # (14 days) time from secondary waning phase until gumbel distribution reaches its min scale parameter
t_LOD = math.inf # ( inf days ) Time from infection until modal read counts value is equal to the limit of detection

sigma_obs = 3.5 #  Initial scale parameter for the Gumbel distribution until a=teclipse+tpeak+tswitch
s_mod = 0.4 #  0.4 multiplicative factor applied to scale paramter for the Gumble distrbution - starting at t_eclipse + t_peak + t_switch + t_scle
c_zero = 40 #  Ct value at time of infection
c_peak = 20 #  (20) Modal Ct value at peak viral load
c_switch = 30 #  (33) Modal Ct value at a = teclipse + tpeak + tswitch
c_LOD = 40 #  Limit of detection of Ct value

parameters_ct = [
    t_eclipse, t_peak, t_switch, t_mod, t_LOD,
    c_zero, c_peak, c_switch, c_LOD,
    s_mod, sigma_obs]

# Set read counts value for the suceptible and recovered individuals
Ct_susc = 40

### Plot Ct model

In [5]:
time_from_infec = np.arange(1, 50)
ct_val = []

for ti in time_from_infec:
    ti_ct_val = []
    for _ in range(10000):
        ti_ct_val.append(algorithm.ct_model(parameters_ct, ti))
    ct_val.append(ti_ct_val)

ct_val = np.asarray(ct_val)

In [6]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=time_from_infec,
        x=np.mean(ct_val, axis=1),
        mode='lines',
        name='Mean Ct value',
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        y=time_from_infec.tolist() + time_from_infec.tolist()[::-1],
        x=np.quantile(ct_val, 0.975, axis=1).tolist() + np.quantile(ct_val, 0.025, axis=1).tolist()[::-1],
        mode='lines',
        fill='toself',
        fillcolor='blue',
        line_color='blue',
        opacity=0.3,
        showlegend=False,
    )
)

# Add axis labels
fig.update_layout(
    width=500, 
    height=500,
    plot_bgcolor='white',
    xaxis=dict(
        linecolor='black',
        title = 'Mean Ct value',
        autorange='reversed'
        ),
    yaxis=dict(
        linecolor='black',
        title = 'Time since infection'),
    #legend=dict(
    #    orientation="h",
    #    yanchor="bottom",
    #    y=1.02,
    #    xanchor="right",
    #    x=1
    #)
    )

fig.write_image('images/Logistic_Ct_value_model.pdf')
fig.show()

### Compute the history of recovered individuals that fully clear the virus and generation times distribution

#### 0 = 'not cleared'; 1 = 'cleared'

In [7]:
# Daily probability of recovered fully clearing the virus
p_addl = 0.2

R_history_clear_algorithm = []

# Go through each run experiment
for _ in range(num_experiments):
    R_history_clear = []

    # Go through each recorded day
    for t, time in enumerate(times):
        current_clear_status = []

        # If there are any recovered individual
        if len(R_times_history_algorithm[_][t]) > 0:
            # Go through each of them and
            for ind, ind_ID in enumerate(R_history_algorithm[_][t]):
                clear_status = 0

                # If they have previously cleared the virus they signal that
                if ind_ID in R_history_algorithm[_][t-1] and R_history_clear[-1][R_history_algorithm[_][t-1].index(ind_ID)] == 1:
                    clear_status = 1
                # if not, they could do it today, if their time since infection exceeds teclipse + tpeak + tswitch
                elif time > R_times_history_algorithm[_][t][ind] + t_eclipse + t_peak + t_switch:
                    clear_status = 1 - np.random.binomial(1, p = (1-p_addl)**(
                        time - R_times_history_algorithm[_][t][ind] - t_eclipse - t_peak - t_switch))

                current_clear_status.append(clear_status)

        R_history_clear.append(current_clear_status)                

    R_history_clear_algorithm.append(R_history_clear)

In [8]:
# Compute the generation times distribution, which also follows a
# right-skewed Gumbel distribution
generation_times = []

for _ in range(70):
    if _ < t_eclipse + t_peak + t_switch:
        generation_times.append(
            gumbel_r.cdf(
                c_LOD,
                algorithm._compute_mode_ct_model(
                    _, t_eclipse, t_peak, t_switch, t_LOD,
                    c_zero, c_peak, c_switch, c_LOD),
                algorithm._compute_sigma_ct_model(
                    _, t_eclipse, t_peak, t_switch, t_mod,
                    s_mod, sigma_obs)
            ))
        
    else:
        generation_times.append(
            gumbel_r.cdf(
                c_LOD,
                algorithm._compute_mode_ct_model(
                    _, t_eclipse, t_peak, t_switch, t_LOD,
                    c_zero, c_peak, c_switch, c_LOD),
                algorithm._compute_sigma_ct_model(
                    _, t_eclipse, t_peak, t_switch, t_mod,
                    s_mod, sigma_obs)
            ) * (1-p_addl)**(_ - t_eclipse - t_peak - t_switch))

#### Plot generation times

In [9]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=time_from_infec,
        y=generation_times,
        mode='lines',
        name='Generation times',
        showlegend=False,
    )
)

fig.show()

## Parameter inference
In this section we test the quality of parameter inference for an optimisation approach, using the Bare-bones CMA-ES method from *Pints [2]* for single parameter inference, for a range of sample sizes and frequencies of sampling values, to compare the quality of parameter estimation and proportion of infected population across different sampling protocols.

#### Sample individuals with specific frequencies and in specific batch sizes

In [10]:
freq_samplying_range = [3, 5, 7, 15, 21, 30]
sample_size_range = [20, 30, 40, 50]

#### Method to create viral read data and ground truth

In [11]:
def sensitivity_analysis_run(sample_points, sample_size):
    ct_values = []
    ct_infec = []

    ct_susc_ids = []
    ct_infec_ids = []
    ct_recov_ids = []

    ct_time_of_recov_infec = []
    ct_time_of_infec = []
    ct_time_since_infec = []

    for _ in range(num_experiments):
        experiment_ct_values = []
        experiment_infec = []

        experiment_susc_ids = []
        experiment_infec_ids = []
        experiment_recov_ids = []

        experiment_time_of_recov_infec = []
        experiment_time_of_infec = []
        experiment_time_since_infec = []
        # At each point in time sample sample_size individuals
        for time in sample_points:
            # Identify the current infections at the specified timepoint
            current_susceptibles = S_history_algorithm[_][time-1]
            current_infections = I_history_algorithm[_][time-1]
            current_recovered = R_history_algorithm[_][time-1]
            current_infection_times = I_times_history_algorithm[_][time-1]
            current_recov_infection_times = R_times_history_algorithm[_][time-1]
            current_recov_clear_virus_status = R_history_clear_algorithm[_][time-1]

            # Sample without replacement the sample_size individuals and
            # determine their time since infection to produce Ct values
            number_selected_susc, number_selected_infec, number_selected_rec = \
                multinomial.rvs(
                    n=sample_size,
                    p=output_algorithm[_, time-1, :]/np.sum(output_algorithm[_, time-1, :])) # determine how many of those sampled are S, I and R

            # First add the Ct values for the sampled susceptibele and recovered individuals
            sampled_ct_values = [Ct_susc] * number_selected_susc

            selected_individuals_susc_ids = np.random.choice(
                    current_susceptibles,
                    size=number_selected_susc,
                    replace=False).tolist() # determine the ids of those sampled Ss
            
            if len(current_recov_infection_times) > 0:
                # If we have at least one selected recovered
                selected_individuals_indices = np.random.choice(
                    range(len(current_recov_infection_times)),
                    size=number_selected_rec,
                    replace=False).tolist() # determine the indices of those sampled Rs
            
                selected_individuals_rec_ids = [current_recovered[_] for _ in selected_individuals_indices]

                # Determine the time of infection of those sampled Rs
                selected_individuals_recov_infec_times = [current_recov_infection_times[_] for _ in selected_individuals_indices]

                sample_time_since_infec = time - selected_individuals_recov_infec_times # determine how long since infection for selected Rs

                # Determine the clearence of infection of those sampled Rs
                selected_individuals_clear_virus_status = [current_recov_clear_virus_status[_] for _ in selected_individuals_indices]

                # Run Ct model to determine individual Ct counts for each sample
                for i, ti in enumerate(sample_time_since_infec):
                    sampled_ct_values.append(algorithm.ct_model(parameters_ct, ti) * selected_individuals_clear_virus_status[i])

            elif number_selected_rec > 0:
                # If initial step when no history of infection is provided
                for i in range(number_selected_rec):
                    sampled_ct_values.append(algorithm.ct_model(parameters_ct, time))
                
                sample_time_since_infec = np.zeros(number_selected_rec)
                selected_individuals_rec_ids = [] 
            else:
                sample_time_since_infec = []
                selected_individuals_rec_ids = []

            if len(current_infection_times) > 0:
                # If we have at least one selected infection
                selected_individuals_indices = np.random.choice(
                    range(len(current_infection_times)),
                    size=number_selected_infec,
                    replace=False).tolist() # determine the indices of those sampled Is
                
                # Determine the ids of those sampled Is
                selected_individuals_infec_ids = [current_infections[_] for _ in selected_individuals_indices]

                # Determine the time of infection of those sampled Is
                selected_individuals_infec_times = [current_infection_times[_] for _ in selected_individuals_indices]
            
                sample_time_since_infec = time - selected_individuals_infec_times # determine how long since infection for selected Is

                # Run Ct model to determine individual Ct counts for each sample
                for ti in sample_time_since_infec:
                    sampled_ct_values.append(algorithm.ct_model(parameters_ct, ti))
            
            elif number_selected_infec > 0:
                # If initial step when no history of infection is provided
                for i in range(number_selected_infec):
                    sampled_ct_values.append(algorithm.ct_model(parameters_ct, time))
                
                selected_individuals_infec_times = np.zeros(number_selected_infec)
                sample_time_since_infec = np.zeros(number_selected_infec)
                selected_individuals_infec_ids = [] 
            else:
                selected_individuals_infec_times = []
                sample_time_since_infec = []
                selected_individuals_infec_ids = [] 

            experiment_ct_values.append(sampled_ct_values)
            experiment_infec.append(number_selected_infec)
            
            experiment_susc_ids.append(selected_individuals_susc_ids)
            experiment_infec_ids.append(selected_individuals_infec_ids)
            experiment_recov_ids.append(selected_individuals_rec_ids)

            experiment_time_of_recov_infec.append(selected_individuals_recov_infec_times)
            experiment_time_of_infec.append(selected_individuals_infec_times)
            experiment_time_since_infec.append(sample_time_since_infec)
        
        ct_values.append(experiment_ct_values)
        ct_infec.append(experiment_infec)

        ct_susc_ids.append(experiment_susc_ids)
        ct_infec_ids.append(experiment_infec_ids)
        ct_recov_ids.append(experiment_recov_ids)

        ct_time_of_recov_infec.append(experiment_time_of_recov_infec)
        ct_time_of_infec.append(experiment_time_of_infec)
        ct_time_since_infec.append(experiment_time_since_infec)

    ct_values = np.asarray(ct_values)
    ct_infec = np.asarray(ct_infec)

    ct_time_of_infec_data = []

    for _ in range(num_experiments):
        experiment_ct_time_of_infec_data = pd.DataFrame(columns=['ID', 'Value'])
        for t, time in enumerate(sample_points):
            experiment_ct_time_of_infec_data = pd.concat(
                [
                    experiment_ct_time_of_infec_data,
                    pd.DataFrame({
                        'ID': ct_susc_ids[_][t] + ct_recov_ids[_][t] + ct_infec_ids[_][t],
                        'Value': [400] * len(ct_susc_ids[_][t]) + ct_time_of_recov_infec[_][t] + ct_time_of_infec[_][t]
                    })
                ])
            
        ct_time_of_infec_data.append(experiment_ct_time_of_infec_data)

    ct_values_data = []

    for _ in range(num_experiments):
        experiment_ct_values_data = pd.DataFrame(columns=['ID', 'TimeOfSample', 'Value'])
        for t, time in enumerate(sample_points):
            experiment_ct_values_data = pd.concat(
                [
                    experiment_ct_values_data,
                    pd.DataFrame({
                        'ID': ct_susc_ids[_][t] + ct_recov_ids[_][t] + ct_infec_ids[_][t],
                        'TimeOfSample': [time] * sample_size,
                        'Value': ct_values[_, t, :].tolist()
                    })
                ])
            
        ct_values_data.append(experiment_ct_values_data)

    mvr_inference = mmi.LogisticGrowthMVRCtValInfer(algorithm, generation_times=generation_times)

    # Read Vireal read counts and Ct values data
    mvr_inference.read_ct_values_data(ct_values_data[0], parameters_ct)

    R0_found = mvr_inference.optimisation_problem_setup()[0]

    shody_recov_freq = []

    for t in range(ct_values[0].shape[0]):
        shody_recov_freq.append((np.where((ct_values[0][t, :] > 29) & (ct_values[0][t, :] < 31))[0]).shape[0] /sample_size)

    return ct_values_data, R0_found, shody_recov_freq, ct_infec[0, :] / sample_size

In [12]:
# Transform birth rate and death rates into function format for inference method
parameters[3] = lambda _: theta
parameters[4] = lambda _: mu
parameters[5] = lambda _: nu

#### Method to run inference with viral read data and plot inferred trajectories against ground truth

In [13]:
def routine_run(freq_samplying, sample_size):
    sample_points = np.arange(20, 200, freq_samplying)

    # For each choice of sample size and frequency infer parameters: 
    ct_values_data, R0_found, shody_recov_freq, infec_freq_sample = sensitivity_analysis_run(sample_points, sample_size)

    mvr_inference = mmi.LogisticGrowthMVRCtValInfer(algorithm, generation_times=generation_times)
    mvr_inference.read_ct_values_data(ct_values_data[0], parameters_ct)
    mvr_inference._create_posterior()

    parameters[6] = R0_found[0] / infect_period

    theta_found = []
    for _ in range(1000):
        output_found = algorithm.simulate_fixed_times(parameters, start_time, end_time)[0]

        theta_found.append(np.divide(output_found[:, 1], np.sum(output_found, axis=1)))

    theta_found = np.array(theta_found)

    theta_found_mean = np.mean(theta_found, axis=0)
    theta_found_upper = np.quantile(theta_found, 0.975, axis=0)
    theta_found_lower = np.quantile(theta_found, 0.025, axis=0)

    output_found = mvr_inference.loglikelihood._run_sir_model(
        parameters, np.arange(max(mvr_inference.loglikelihood._ct_sampled_times))
    )
    theta_found_det = np.divide(output_found[:, 1], np.sum(output_found, axis=1))

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            y=(output_algorithm[0, :, 1] / np.mean(np.sum(output_algorithm, axis=2), axis=0)).tolist(),
            x=times,
            mode='lines',
            name='True freq',
            line_color='red'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=infec_freq_sample.tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq',
            line_color='blue'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=(1 - np.array(shody_recov_freq)).tolist(),
            x=sample_points,
            mode='lines',
            name='Sample freq (extreme I)',
            line_color='green'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_det.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq - det',
            line_color='black',
            line_dash='dash'
        )
    )

    fig.add_trace(
        go.Scatter(
            y=theta_found_mean.tolist(),
            x=times,
            mode='lines',
            name='Inferred freq',
            line_color='black'
        )
    )

    fig.add_trace(
        go.Scatter(
            x=times + times[::-1],
            y=theta_found_upper.tolist() + theta_found_lower.tolist()[::-1],
            fill='toself',
            fillcolor='black',
            line_color='black',
            opacity=0.3,
            mode='lines',
            showlegend=False,
            name='Inferred freq'
        )
    )

    # Add axis labels
    fig.update_layout(
        title='Sensitivity analysis: Frequency:{} days; Sample size:{}'.format(freq_samplying, sample_size),
        width=700, 
        height=400,
        plot_bgcolor='white',
        xaxis=dict(
            linecolor='black',
            title = 'Time (days)'
            ),
        yaxis=dict(
            linecolor='black',
            title = 'Individuals'),
        )

    fig.write_image('images/Logistic_Ct_CredInt_Freq_{}_Sample_size_{}.pdf'.format(freq_samplying, sample_size))
    fig.show()

#### Run optimisation-based inference method for multiple sampling protcols

In [14]:
routine_run(freq_samplying_range[2], sample_size_range[0])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_70947/145446728.py:143: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_70947/145446728.py:159: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Iter. Eval. Best      Current   Time    
0     4     -3622.306 -3622.306   0:15.1
1     8     -3597.037 -3597.037   0:29.9
2     12    -3596.318 -3596.318   0:46.7
3     16    -3596.318 -3604.951   0:58.4
20    84    -3594.817 -3594.817   6:24.8
40    164   -3594.817 -3594.817  13:25.4
60    244   -3594.817 -3594.817  20:19.0
80    324   -3594.817 -3594.817  27:11.0
100   404   -3594.817 -3594.817  33:56.1
105   420   -3594.817 -3594.817  35:16.9
Halting: No significant change in best function evaluation for 100 iterations.
[2.09650496] -3594.816683144401
Optimisation phase is finished.


In [15]:
routine_run(freq_samplying_range[2], sample_size_range[3])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_70947/145446728.py:143: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_70947/145446728.py:159: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Iter. Eval. Best      Current   Time    
0     4     -6837.285 -6837.285   0:53.8
1     8     -6826.765 -6826.765   1:31.7
2     12    -6810.836 -6810.836   2:21.4
3     16    -6810.836 -6822.605   3:12.0
20    84    -6808.535 -6808.535  17:29.8
40    164   -6808.535 -6808.535  32:56.0
60    244   -6808.535 -6808.535  51:50.0
80    324   -6808.535 -6808.535  71:50.6
100   404   -6808.535 -6808.535  91:50.4
106   424   -6808.535 -6808.535  95:54.9
Halting: No significant change in best function evaluation for 100 iterations.
[2.0735367] -6808.534642254052
Optimisation phase is finished.


In [16]:
routine_run(freq_samplying_range[5], sample_size_range[0])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_70947/145446728.py:143: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_70947/145446728.py:159: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Iter. Eval. Best      Current   Time    
0     4     -470.4677 -470.4677   0:02.6
1     8     -466.3075 -466.3075   0:06.3
2     12    -466.3075 -467.907    0:09.9
3     16    -465.9438 -465.9438   0:13.2
20    84    -465.2646 -465.2646   1:08.6
40    164   -465.2646 -465.2646   2:12.3
60    244   -465.2646 -465.2646   3:16.8
80    324   -465.2646 -465.2646   4:20.8
100   404   -465.2646 -465.2646   5:25.6
106   424   -465.2646 -465.2646   5:41.9
Halting: No significant change in best function evaluation for 100 iterations.
[2.15325329] -465.26456282813115
Optimisation phase is finished.


In [17]:
routine_run(freq_samplying_range[5], sample_size_range[3])

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_70947/145446728.py:143: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/var/folders/ph/jyxnc9y52svgq2k5lt2q4r000000gp/T/ipykernel_70947/145446728.py:159: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/Users/ioaros/opt/anaconda3/envs/metavirom/lib/python3.13/site-packages/pints/_optimisers/__init__.py:988: UserWarning:

The method `set_max_unchanged_iterations` is deprecated. Please use `set_function_tolerance` instead.



Maximising LogPDF
Using Bare-bones CMA-ES
Running in sequential mode.
Population size: 4
Iter. Eval. Best      Current   Time    
0     4     -2646.184 -2646.184   0:04.2
1     8     -2618.96  -2618.96    0:12.3
2     12    -2618.96  -2653.679   0:19.8
3     16    -2618.96  -2620.398   0:27.6
20    84    -2585.892 -2585.892   2:32.2
40    164   -2585.891 -2585.891   4:46.9
60    244   -2585.891 -2585.891   6:52.3
80    324   -2585.891 -2585.891   8:56.1
100   404   -2585.891 -2585.891  11:01.3
107   428   -2585.891 -2585.891  11:39.7
Halting: No significant change in best function evaluation for 100 iterations.
[2.12602334] -2585.891462322315
Optimisation phase is finished.
